# Question 1

Build a personalized FAQ knowledge base.

In [1]:
import pandas as pd

rollNo = "1024170231"
digits = [int(digit) for digit in rollNo]
last_two_digits = digits[-2:]

categories = ["billing", "account", "general"]
last_two_digits


[3, 1]

In [2]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]


In [3]:
personalized_details = {
    "billing": {
        "question": "how can i download a duplicate payment receipt",
        "answer": "Open Billing > Payment History and select Download Receipt.",
        "keywords": "receipt duplicate payment invoice",
    },
    "account": {
        "question": "how do i update my registered mobile number",
        "answer": "Go to Account Settings and edit your mobile number.",
        "keywords": "update mobile number registered",
    },
    "general": {
        "question": "where can i find the help center",
        "answer": "Open the Help section from the main menu.",
        "keywords": "help center support guide",
    },
}

personalized_entries = []
for digit in last_two_digits:
    category = categories[digit % 3]
    entry = personalized_details[category].copy()
    entry["category"] = category
    personalized_entries.append(entry)

personalized_entries


[{'question': 'how can i download a duplicate payment receipt',
  'answer': 'Open Billing > Payment History and select Download Receipt.',
  'keywords': 'receipt duplicate payment invoice',
  'category': 'billing'},
 {'question': 'how do i update my registered mobile number',
  'answer': 'Go to Account Settings and edit your mobile number.',
  'keywords': 'update mobile number registered',
  'category': 'account'}]

In [4]:
faq_df = pd.DataFrame(fixed_entries + personalized_entries)
faq_df


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i download a duplicate payment receipt,Open Billing > Payment History and select Down...,receipt duplicate payment invoice,billing
5,how do i update my registered mobile number,Go to Account Settings and edit your mobile nu...,update mobile number registered,account


# Question 2

Generate and score a hypothesis for a query.

In [5]:
import re


def get_words(text):
    return set(re.findall(r"\b\w+\b", text.lower()))


def score_hypothesis(query, df):
    query_words = get_words(query)
    matches = []

    for _, entry in df.iterrows():
        entry_words = get_words(entry["question"] + " " + entry["keywords"])
        matched_words = query_words.intersection(entry_words)

        if matched_words:
            matches.append({
                "question": entry["question"],
                "answer": entry["answer"],
                "category": entry["category"],
                "matched_keywords": ", ".join(sorted(matched_words)),
                "confidence": round(len(matched_words) / len(query_words) * 100, 2),
            })

    result = pd.DataFrame(matches)
    if result.empty:
        return result

    return result.sort_values(
        by=["confidence", "question"], ascending=[False, True], ignore_index=True
    )


score_hypothesis("how do i reset my password", faq_df)


,question,answer,category,matched_keywords,confidence
0,how do i update my registered mobile number,Go to Account Settings and edit your mobile nu...,account,"do, how, i, my",66.67
1,how to reset password,Go to Settings > Reset Password.,account,"how, password, reset",50.00
2,how can i download a duplicate payment receipt,Open Billing > Payment History and select Down...,billing,"how, i",33.33
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,"how, i",33.33


# Question 3

Find all questions in the category of a personalized entry.

In [6]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]


personalized_category = personalized_entries[0]["category"]
same_category(personalized_category, faq_df)


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i download a duplicate payment receipt,Open Billing > Payment History and select Down...,receipt duplicate payment invoice,billing


# Question 4

Add a new keyword to one FAQ entry and save the updated knowledge base.

In [7]:
selected_index = 4
selected_question = faq_df.at[selected_index, "question"]
new_keyword = input(f"Enter a new keyword for: {selected_question}\n").strip().lower()

if new_keyword:
    existing_keywords = faq_df.at[selected_index, "keywords"].split()
    if new_keyword not in existing_keywords:
        faq_df.at[selected_index, "keywords"] += f" {new_keyword}"

output_file = f"{rollNo}_faq_data.csv"
faq_df.to_csv(output_file, index=False)
print(f"Updated FAQ data saved to {output_file}")
faq_df


Updated FAQ data saved to 1024170231_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i download a duplicate payment receipt,Open Billing > Payment History and select Down...,receipt duplicate payment invoice how,billing
5,how do i update my registered mobile number,Go to Account Settings and edit your mobile nu...,update mobile number registered,account


# Question 5

Count FAQ entries in each category.

In [8]:
faq_df.groupby("category").size().reset_index(name="faq_count")


,category,faq_count
0,account,2
1,billing,3
2,general,1


# Question 6

Display every highest-scoring FAQ when there is a tie.

In [9]:
def highest_scoring_matches(query, df):
    ranked_matches = score_hypothesis(query, df)

    if ranked_matches.empty:
        print("No matching FAQ entries found.")
        return ranked_matches

    highest_score = ranked_matches["confidence"].max()
    best_matches = ranked_matches[ranked_matches["confidence"] == highest_score]

    if len(best_matches) > 1:
        print(f"Tie found: {len(best_matches)} entries have confidence {highest_score}%.")
    else:
        print(f"One best match found with confidence {highest_score}%.")

    return best_matches


### Query that produces a tie

In [10]:
highest_scoring_matches("fee", faq_df)


Tie found: 2 entries have confidence 100.0%.


,question,answer,category,matched_keywords,confidence
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,fee,100.0
1,what is the annual fee,The annual fee is Rs 500.,billing,fee,100.0


### Query that produces one best match

In [11]:
highest_scoring_matches("reset password", faq_df)


One best match found with confidence 100.0%.


,question,answer,category,matched_keywords,confidence
0,how to reset password,Go to Settings > Reset Password.,account,"password, reset",100.0
